# Práctica 2 — Media-varianza y modelos multifactoriales

**Pregunta de fondo:** ¿lo que parece alfa lo sigue pareciendo cuando se controla por factores?

* **Parte A** (`quantmgmt.optimizers`): frontera eficiente de Markowitz con restricciones realistas, referencias (equiponderada, SPY, SML del CAPM) y prueba fuera de muestra.
* **Parte B** (`quantmgmt.analytics`): descomposición de la rentabilidad en primas de riesgo con CAPM, FF3, Carhart y FF5.

**Universo:** 10 ETFs sectoriales de EE. UU. más SPY, datos mensuales desde 2010.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from matplotlib.ticker import PercentFormatter

from quantmgmt.data import YahooFinanceData, excess_returns, get_factors, monthly_returns
from quantmgmt.optimizers import (
    Constraints,
    effective_n,
    efficient_frontier,
    estimate_inputs,
    max_sharpe,
    min_variance,
    portfolio_return,
    portfolio_volatility,
)
from quantmgmt.analytics import compare_models, rolling_betas, run_factor_regression
from quantmgmt.utils import generate_risk_summary

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 200)

## 0. Datos

* **ETFs:** precios mensuales ajustados por dividendos (`Adj Close`) de yfinance, con caché en `data/cache/`. Se descarta el mes en curso si está incompleto.
* **Factores:** Ken French Data Library vía `pandas_datareader`. Vienen en % y ya en exceso sobre el tipo libre de riesgo: se pasan a tanto por uno, y a los ETFs se les resta el `RF` del mismo fichero.
* **Fechas:** todo se indexa por mes (`PeriodIndex`) para que ETFs y factores casen sin desfases.

In [ ]:
SECTORS = ["XLK", "XLF", "XLE", "XLV", "XLI", "XLY", "XLP", "XLU", "XLB", "XLRE"]
BENCHMARK = "SPY"
TODAY = pd.Timestamp.today().strftime("%Y-%m-%d")

prices = YahooFinanceData().get_prices(SECTORS + [BENCHMARK], "2010-01-01", TODAY, interval="1mo")
returns = monthly_returns(prices)[SECTORS + [BENCHMARK]]   # retornos totales
factors = get_factors(start="2010-01")
rf = factors["ff3"]["RF"]

ex = excess_returns(returns, rf)                              # retornos en exceso
returns = returns.loc[ex.index]                               # mismos meses que los factores
RF_ANNUAL = rf.loc[ex.index].mean() * 12

print(f"Meses: {ex.index[0]} -> {ex.index[-1]} ({len(ex)})   rf medio anual: {RF_ANNUAL:.2%}")
ex.count().to_frame("meses con dato").T

**Decisión sobre XLRE.** XLRE empieza a cotizar en octubre de 2015. Para optimizar hace falta una covarianza con todos los activos en los mismos meses (si no, la matriz puede no ser semidefinida positiva y el optimizador da resultados sin sentido). Opciones: recortar toda la muestra a 2015-2026 o excluir XLRE de la optimización.

Se **excluye XLRE de la Parte A** (optimización) y se mantiene en la SML y en la Parte B, donde cada regresión usa los meses disponibles de cada activo. Así la Parte A conserva la ventana 2010-2017 que exige la prueba fuera de muestra, y XLRE solo pierde el papel de candidato en la cartera.

In [ ]:
OPT = [s for s in SECTORS if s != "XLRE"]   # universo de la optimización (9 sectores)

ex_opt = ex[OPT].dropna()
mu, cov = estimate_inputs(ex_opt, periods_per_year=12)   # en exceso -> rf = 0 en el Sharpe

pd.DataFrame({
    "Rentabilidad exceso (anual)": mu,
    "Volatilidad (anual)": np.sqrt(np.diag(cov)),
}, index=OPT).style.format("{:.1%}")

## Parte A — Optimización media-varianza

### A.1 Frontera eficiente: efecto de las restricciones

Tres fronteras sobre la muestra completa:

* **Con cortos**: sin restricciones de signo.
* **Long-only**: sin cortos, sin límite por activo.
* **Long-only con máximo 20 %** por activo.

La concentración se mide con el **número efectivo de activos**, $N_{ef} = 1/\sum w_i^2$: vale 9 si la cartera es equiponderada entre los 9 sectores y 1 si todo está en uno.

In [ ]:
SETUPS = {
    "Con cortos": Constraints(long_only=False),
    "Long-only": Constraints(),
    "Long-only, máx. 20 %": Constraints(max_weight=0.20),
}

frontiers = {name: efficient_frontier(mu, cov, n_points=40, constraints=c) for name, c in SETUPS.items()}
tangency = {name: max_sharpe(mu, cov, constraints=c) for name, c in SETUPS.items()}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
for name, f in frontiers.items():
    ax1.plot(f["volatility"], f["return"], label=name)
    w = tangency[name]
    ax1.scatter(portfolio_volatility(w, cov), portfolio_return(w, mu), s=80, marker="*", zorder=5)
    ax2.plot(f["return"], f["effective_n"], label=name)

ax1.scatter(np.sqrt(np.diag(cov)), mu, color="black", s=15)
for t in OPT:
    ax1.annotate(t, (np.sqrt(cov.loc[t, t]), mu[t]), fontsize=8, xytext=(3, 3), textcoords="offset points")
ax1.set(xlabel="Volatilidad anual", ylabel="Rentabilidad en exceso anual", title="Fronteras eficientes (estrella = máximo Sharpe)")
ax2.set(xlabel="Rentabilidad en exceso anual", ylabel="Nº efectivo de activos", title="Concentración a lo largo de la frontera")
for ax in (ax1, ax2):
    ax.legend()
ax1.xaxis.set_major_formatter(PercentFormatter(1.0)); ax1.yaxis.set_major_formatter(PercentFormatter(1.0))
ax2.xaxis.set_major_formatter(PercentFormatter(1.0))
plt.show()

In [ ]:
weights_table = pd.DataFrame(tangency).T
weights_table["Sharpe"] = [portfolio_return(w, mu) / portfolio_volatility(w, cov) for w in tangency.values()]
weights_table["N efectivo"] = [effective_n(w) for w in tangency.values()]
weights_table["Activos con peso > 1 %"] = [(w > 0.01).sum() for w in tangency.values()]

weights_table.style.format({**{t: "{:.1%}" for t in OPT}, "Sharpe": "{:.2f}", "N efectivo": "{:.1f}"})

**Comentario:**

_(Completar. Qué mirar: con cortos aparecen posiciones extremas, largas y cortas, que no tienen sentido económico; long-only concentra en pocos sectores, sobre todo al subir por la frontera, donde acaba en un solo activo; el límite del 20 % obliga a repartir, sube el N efectivo y la frontera se desplaza a la derecha, es decir, se paga algo de Sharpe a cambio de diversificación.)_

**Restricción de rotación.** Ejemplo: partiendo de la equiponderada, ¿cómo cambia la cartera de máximo Sharpe si solo se permite rotar un 20 % o un 50 %? (Rotación = suma de compras y ventas en valor absoluto.)

In [ ]:
ew_opt = pd.Series(1 / len(OPT), index=OPT)

turnover_table = pd.DataFrame({
    f"Rotación máx. {L:.0%}": max_sharpe(mu, cov, constraints=Constraints(prev_weights=ew_opt, max_turnover=L))
    for L in (0.2, 0.5)
}).T
turnover_table.loc["Sin límite"] = tangency["Long-only"]
turnover_table["Rotación usada"] = (turnover_table[OPT] - ew_opt).abs().sum(axis=1)
turnover_table["Sharpe"] = [portfolio_return(w, mu) / portfolio_volatility(w, cov) for w in turnover_table[OPT].to_numpy()]

turnover_table.style.format({**{t: "{:.1%}" for t in OPT}, "Rotación usada": "{:.0%}", "Sharpe": "{:.2f}"})

### A.2 Máximo Sharpe frente a equiponderada y SPY

Se compara la cartera de máximo Sharpe **long-only con límite del 20 %** (la versión realista) con la equiponderada de los 9 sectores y con SPY, usando las funciones de la Práctica 1 sobre datos mensuales (`periods_per_year=12`).

Ojo: en la muestra completa la comparación está sesgada a favor del máximo Sharpe, que se ha optimizado con esos mismos datos. La comparación honesta es la fuera de muestra (A.4).

In [ ]:
def nav(returns: pd.DataFrame, weights: pd.Series) -> pd.Series:
    """NAV (base 100) de una cartera rebalanceada cada mes a pesos constantes."""
    r = returns[weights.index].dropna().dot(weights)
    start = pd.Series([100.0], index=[r.index[0] - 1])
    return pd.concat([start, 100 * (1 + r).cumprod()])


W_MS = tangency["Long-only, máx. 20 %"]

navs = pd.DataFrame({
    "Máx. Sharpe (20 %)": nav(returns, W_MS),
    "Equiponderada": nav(returns, ew_opt),
    "SPY": nav(returns, pd.Series({BENCHMARK: 1.0})),
})

COLS = ["CAGR", "Annualized Volatility", "Sharpe Ratio", "Sortino Ratio",
        "Maximum Drawdown", "Calmar Ratio", "Max Time Under Water"]
FMT = {"CAGR": "{:.2%}", "Annualized Volatility": "{:.2%}", "Sharpe Ratio": "{:.2f}",
       "Sortino Ratio": "{:.2f}", "Maximum Drawdown": "{:.2%}", "Calmar Ratio": "{:.2f}",
       "Max Time Under Water": "{:.0f}"}

full_summary = generate_risk_summary(navs, periods_per_year=12, risk_free_rate=RF_ANNUAL)
navs.set_index(navs.index.to_timestamp()).plot(logy=True, title="NAV, muestra completa (base 100, escala log)")
plt.show()
full_summary[COLS].style.format(FMT)

**Comentario:**

_(Completar.)_

### A.3 SML del CAPM

Para cada sector: beta contra SPY (con retornos en exceso) y rentabilidad media en exceso anualizada. Si el CAPM se cumple, los puntos están sobre la **SML teórica**: recta que pasa por el origen con pendiente igual a la prima de mercado (rentabilidad en exceso media de SPY). Se estima la recta empírica $\bar r_i = \gamma_0 + \gamma_1 \beta_i$ y se contrasta $\gamma_0 = 0$ y $\gamma_1 = $ prima de mercado.

In [ ]:
def capm_beta(y: pd.Series, x: pd.Series) -> float:
    data = pd.concat([y, x], axis=1).dropna()
    return sm.OLS(data.iloc[:, 0], sm.add_constant(data.iloc[:, 1])).fit().params.iloc[1]


sml = pd.DataFrame({
    "beta": {s: capm_beta(ex[s], ex[BENCHMARK]) for s in SECTORS},
    "rentabilidad exceso": ex[SECTORS].mean() * 12,
})
market_premium = ex[BENCHMARK].mean() * 12

fit = sm.OLS(sml["rentabilidad exceso"], sm.add_constant(sml["beta"])).fit()
gamma0, gamma1 = fit.params
t0, t1 = fit.tvalues
t_slope_vs_theory = (gamma1 - market_premium) / fit.bse["beta"]

print(f"Prima de mercado (SPY): {market_premium:.2%}")
print(f"γ0 = {gamma0:.2%} (t = {t0:.2f})   teoría: 0")
print(f"γ1 = {gamma1:.2%} (t = {t1:.2f})   teoría: {market_premium:.2%}  (t vs teoría = {t_slope_vs_theory:.2f})")
print(f"R² = {fit.rsquared:.2f}")

b = np.linspace(0, sml["beta"].max() * 1.1, 50)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(sml["beta"], sml["rentabilidad exceso"])
for s, row in sml.iterrows():
    ax.annotate(s, (row["beta"], row["rentabilidad exceso"]), xytext=(4, 4), textcoords="offset points")
ax.plot(b, market_premium * b, "k--", label="SML teórica")
ax.plot(b, gamma0 + gamma1 * b, "tab:red", label="Recta estimada")
ax.scatter(1, market_premium, marker="*", s=200, color="black", label="SPY")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set(xlabel="Beta contra SPY", ylabel="Rentabilidad media en exceso (anual)", title="Security Market Line")
ax.legend()
plt.show()

**Comentario:**

_(Completar. Qué mirar: si la recta estimada es más plana que la teórica, con intercepto positivo, es el resultado clásico (Black, Jensen y Scholes, 1972; Frazzini y Pedersen, 2014): los activos de beta baja rinden más de lo que predice el CAPM y los de beta alta, menos. Con solo 10 puntos y betas concentradas en torno a 1, el contraste tiene poca potencia.)_

### A.4 Fuera de muestra: estimar en 2010-2017, evaluar en 2019-hoy

Se estiman `mu` y `cov` solo con 2010-2017, se construyen las carteras y se mantienen esos pesos (rebalanceo mensual) desde enero de 2019. 2018 queda fuera, como pide el enunciado. Se compara lo **esperado** (lo que prometía la optimización) con lo **obtenido**.

In [ ]:
ins = ex_opt.loc["2010-01":"2017-12"]
mu_in, cov_in = estimate_inputs(ins, periods_per_year=12)

oos_weights = {
    "Máx. Sharpe (long-only)": max_sharpe(mu_in, cov_in),
    "Máx. Sharpe (20 %)": max_sharpe(mu_in, cov_in, constraints=Constraints(max_weight=0.20)),
    "Mín. varianza": min_variance(cov_in),
    "Equiponderada": ew_opt,
}

oos_returns = returns.loc["2019-01":]
oos_ex = ex.loc["2019-01":]

rows = {}
for name, w in oos_weights.items():
    realized = oos_ex[OPT].dot(w)
    rows[name] = {
        "Exceso esperado": portfolio_return(w, mu_in),
        "Exceso obtenido": realized.mean() * 12,
        "Vol. esperada": portfolio_volatility(w, cov_in),
        "Vol. obtenida": realized.std() * np.sqrt(12),
        "Sharpe esperado": portfolio_return(w, mu_in) / portfolio_volatility(w, cov_in),
        "Sharpe obtenido": realized.mean() / realized.std() * np.sqrt(12),
        "N efectivo": effective_n(w),
    }
spy_oos = oos_ex[BENCHMARK]
rows["SPY"] = {"Exceso obtenido": spy_oos.mean() * 12, "Vol. obtenida": spy_oos.std() * np.sqrt(12),
               "Sharpe obtenido": spy_oos.mean() / spy_oos.std() * np.sqrt(12)}

expected_vs_realized = pd.DataFrame(rows).T
expected_vs_realized.style.format({c: "{:.1%}" for c in expected_vs_realized.columns if "Sharpe" not in c and "N " not in c} |
                                  {"Sharpe esperado": "{:.2f}", "Sharpe obtenido": "{:.2f}", "N efectivo": "{:.1f}"}, na_rep="—")

In [ ]:
pd.DataFrame(oos_weights).T.style.format("{:.1%}").set_caption("Pesos estimados con 2010-2017")

In [ ]:
oos_navs = pd.DataFrame({name: nav(oos_returns, w) for name, w in oos_weights.items()})
oos_navs["SPY"] = nav(oos_returns, pd.Series({BENCHMARK: 1.0}))

oos_navs.set_index(oos_navs.index.to_timestamp()).plot(title="Fuera de muestra: NAV desde 2019 (base 100)")
plt.show()

generate_risk_summary(oos_navs, periods_per_year=12, risk_free_rate=RF_ANNUAL)[COLS].style.format(FMT)

**Comentario:**

_(Completar. Qué mirar: cuánto se desvía la rentabilidad obtenida de la esperada, si la cartera más concentrada (long-only sin límite) es la que más se aleja, y si la equiponderada o la de mínima varianza, que no dependen de `mu` o dependen menos, aguantan mejor. Idea clave: Markowitz trata `mu` como conocida, pero es la estimación más ruidosa y el optimizador amplifica sus errores, error maximization en palabras de Michaud.)_

## Parte B — Modelos multifactoriales

`run_factor_regression` estima $r_i - r_f = \alpha + \sum_k \beta_k F_k + \varepsilon$ con errores estándar de Newey-West (robustos a heterocedasticidad y autocorrelación). Se aplica a los 10 sectores y a las carteras de la Parte A:

* **Máx. Sharpe (in-sample)**: pesos de A.2, estimados con toda la muestra. Su alfa está inflada por construcción, porque los pesos se eligieron sabiendo qué sectores iban a rendir más.
* **Máx. Sharpe (OOS)**: pesos estimados con 2010-2017 y aplicados desde 2019, sin mirar al futuro.
* **Equiponderada**.

In [ ]:
portfolios_ex = pd.DataFrame({
    "Máx. Sharpe (in-sample)": ex[OPT].dot(W_MS),
    "Máx. Sharpe (OOS)": oos_ex[OPT].dot(oos_weights["Máx. Sharpe (20 %)"]),
    "Equiponderada": ex[OPT].dot(ew_opt),
})
assets_ex = ex[SECTORS].join(portfolios_ex)

table = compare_models(assets_ex, factors)

def stars(t):
    return "***" if abs(t) > 2.58 else "**" if abs(t) > 1.96 else "*" if abs(t) > 1.65 else ""

alpha_view = pd.DataFrame({
    model: [f"{a:+.2%}{stars(t)}" for a, t in zip(table[(model, "alpha (ann.)")], table[(model, "t(alpha)")])]
    for model in ["CAPM", "FF3", "Carhart", "FF5"]
}, index=table.index)
r2_view = table.xs("adj R2", axis=1, level=1)

print("Alfa anualizada (* p<0.10, ** p<0.05, *** p<0.01, Newey-West)")
alpha_view

In [ ]:
r2_view.style.format("{:.2f}").background_gradient(axis=None, cmap="Greens").set_caption("R² ajustado")

In [ ]:
ff5_full = run_factor_regression(assets_ex, factors, "FF5")
beta_cols = [c for c in ff5_full.columns if c.startswith("beta")]
ff5_full[beta_cols + ["adj R2"]].astype(float).style.format("{:.2f}").background_gradient(
    subset=beta_cols, cmap="RdBu", vmin=-1.5, vmax=1.5).set_caption("Betas FF5")

### Betas rolling (36 meses)

XLK (tecnología) y XLE (energía) tienen exposiciones opuestas al factor valor: la tecnología es el ejemplo clásico de *growth* (HML negativo) y la energía de *value* (HML positivo).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)
for sector, color in [("XLK", "tab:blue"), ("XLE", "tab:orange")]:
    rb = rolling_betas(ex[sector], factors, "FF3", window=36)
    rb.index = rb.index.to_timestamp()
    for ax, factor in zip(axes, ["Mkt-RF", "SMB", "HML"]):
        ax.plot(rb.index, rb[factor], color=color, label=sector)
for ax, factor in zip(axes, ["Mkt-RF", "SMB", "HML"]):
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(f"Beta {factor} (FF3, ventana 36 meses)")
    ax.legend()
plt.tight_layout()
plt.show()

## Discusión (máx. 500 palabras)

**¿Sobrevive alguna alfa al pasar de CAPM a FF5?**

_(Completar.)_

**¿Qué gana el modelo de cinco factores sobre el de tres en este universo?**

_(Completar.)_

**¿Cuánto del resultado de la cartera óptima es exposición factorial disfrazada de habilidad?**

_(Completar.)_

## Conclusiones

_(Completar.)_